# 🐍 Clase 17: Datos — tu neurona entra a una base de datos 🗄️

**Objetivo de la clase:** convertir el CSV crudo en una base de datos consultable, destilar
las **features** que alimentarán al modelo, y hablarle a tu primera **API** — todo dentro de
tu repositorio, con commits.

## 🗺️ Mapa de la clase

| Etapa | Tema | ¿Qué sabrás hacer al terminar? |
|---|---|---|
| **1** | El diccionario de datos | Documentar un dataset con honestidad: lo que se sabe y lo que no |
| **2** | `construir_bd.py` | Leer y correr un script ajeno que usa TU código |
| **3** | SQL con gemelas de pandas | Preguntarle a una base de datos en su idioma |
| **4** | Features: tasa por época | Construir la tabla que decidirá si «la neurona sabía» |
| **5** | La API de GitHub | Pedirle datos a la nube, con token y sin desperdicio |

## 🔗 ¿Recuerdas dónde nos quedamos?

La semana pasada tu proyecto cobró vida: repo, entorno, pruebas y tu primer `Closes #1`.
De tarea quedaron los workers 2 y 3 — la primera celda te dice cómo vas. Y sigue en el
aire el pagaré que dejó el profe de los cuadernos: **¿la neurona sabía?** 💸 Hoy
construimos los datos para poder preguntárselo en serio la próxima clase.

In [ ]:
# Punto de control · ¿cómo llegó tu tarea? ✅
from neurona import Cargador

CSV = "datos/neu_400_RR034075_002_OCPSLA_10_00_VPCizq.csv"
try:
    c = Cargador(CSV)
    print(f"✅ Cargador vivo: {c.metadatos().shape[0]} ensayos, {len(c.tiempos_de_espiga())} trenes.")
    print("   (corre `pytest` en la terminal para ver el resto de la tarea)")
except NotImplementedError as pista:
    print("⏳ Tu Cargador sigue crudo — HOY lo necesitas (la base de datos se construye con él).")
    print("   Pista:", pista)
    print("   Plan: resuélvelo en la Etapa 2 con tu pareja; el código ya lo escribiste el miércoles.")

---
# 🟢 Etapa 1 — El diccionario de datos

*📽 Vienes de AGL-01: trabajo honesto, por escrito, con lo desconocido marcado.*

**En parejas.** El CSV tiene 31 columnas de metadatos y solo conocemos algunas. Un
**diccionario de datos** es el documento que le dice al siguiente ser humano (tú, en dos
meses) qué es cada columna — **y cuáles no sabemos**: eso también es información.

1. Abran **`docs/diccionario-de-datos.md`** (ya tiene la tabla lista para llenar).
2. Usen las dos celdas-lupa de abajo para descifrar las columnas **0–4** y **20–30**.
3. Las columnas **5–19** se quedan como están: *«desconocida — preguntar al laboratorio»*.
   Inventarles significado sería peor que no saber.
4. Al terminar: `git add docs/ · git commit -m "Closes #4: diccionario de datos honesto" · git push`
   *(abre antes el issue #4 en tu repo: «docs · diccionario de datos»)*.

In [ ]:
# 🔍 Lupa 1 · columnas 0–4: enteras y con pocos valores — ¿qué podrían ser?
import numpy as np, pandas as pd

try:
    meta = Cargador(CSV).metadatos()
    df = pd.DataFrame(meta)
    for j in range(5):
        v = np.unique(meta[:, j])
        muestra = v[:8].astype(int) if len(v) < 20 else v[:4]
        print(f"col{j}: {len(v):>3} valores distintos · ejemplo: {muestra}")
    print("\nPistas: una cuenta ensayos… una va de 1 a 14… dos hablan de botones… una es 0/1.")
except NotImplementedError:
    print("⏳ Necesita tu Cargador (ver el punto de control de arriba).")

In [ ]:
# 🔍 Lupa 2 · columnas 20–30: tiempos que siempre crecen — los EVENTOS del ensayo
try:
    fila = meta[0]
    print("Ensayo 1, columnas 20–30 (segundos):")
    print(np.round(fila[20:31], 3))
    print("\n¿Ves que van en orden? Son la cronología de la tarea: inicio, estímulos,")
    print("respuesta, recompensa… El notebook de tasa de disparo ya te enseñó a leerla.")
    print("Compara varios ensayos: ¿cuáles columnas cambian entre ensayos y cuáles casi no?")
except NameError:
    print("⏳ Corre primero la Lupa 1.")

*▶ Volvemos a las diapositivas: BD-01 — por qué tu CSV pide DOS tablas.*

---
# 🟢 Etapa 2 — Construir la base de datos

*📽 Vienes de BD-01.*

**Primero LEE, luego corre.** Abre `scripts/construir_bd.py` y léelo completo (cabe en una
pantalla). Fíjate en tres cosas:

1. **Importa TU `Cargador`** — el worker que escribiste el miércoles hoy es infraestructura. 🤯
2. La tabla `ensayos` nombra solo las columnas **que el diccionario conoce**; las demás
   quedan como `col5`, `col6`… — tu honestidad de la Etapa 1, en código.
3. La tabla `espigas` es el «formato largo»: una fila POR ESPIGA con su número de ensayo.

Luego, en la terminal (con `(.venv)`):

```
python scripts/construir_bd.py
```

**Debes ver:** `ensayos: 140 filas × 31 · espigas: ~20 068 × 2`. Y después corre
`git status`: el `neurona.sqlite` **no aparece** — lo regenerable no se versiona; la receta sí.

*▶ Volvemos a las diapositivas: BD-02 — el idioma SQL y sus gemelas.*

---
# 🟡 Etapa 3 — Las cinco consultas gemelas

*📽 Vienes de BD-02.*

**La regla del bloque:** cada consulta se hace DOS veces — en SQL y en pandas — y las
gemelas deben dar **exactamente lo mismo**. Si difieren, una miente. Las consultas 1 y 2
vienen resueltas; la 3, la 4 y la 5 son tuyas *(soluciones en el apéndice 🔑, para después)*.

In [ ]:
# Gemelas 1 y 2 — resueltas: conteo por clase, y % de aciertos por clase
import sqlite3
from pathlib import Path

if not Path("neurona.sqlite").exists():
    print("⏳ Primero la Etapa 2: python scripts/construir_bd.py")
else:
    con = sqlite3.connect("neurona.sqlite")
    ensayos = pd.read_sql("SELECT * FROM ensayos", con)

    print("— 1 · ¿cuántos ensayos por clase? —")
    sql1 = pd.read_sql("SELECT clase, COUNT(*) AS n FROM ensayos GROUP BY clase", con)
    pan1 = ensayos.groupby("clase").size().rename("n").reset_index()
    print(sql1.head(3), "\n(gemela pandas idéntica:", (sql1["n"] == pan1["n"]).all(), ")")

    print("\n— 2 · % de aciertos por clase —")
    sql2 = pd.read_sql("SELECT clase, AVG(acierto) AS p FROM ensayos GROUP BY clase", con)
    pan2 = ensayos.groupby("clase")["acierto"].mean().rename("p").reset_index()
    print(sql2.head(3), "\n(gemelas iguales:", np.allclose(sql2["p"], pan2["p"]), ")")

In [ ]:
# 🧠 Gemela 3 — TUYA: ¿cuántas espigas tiene cada ensayo? (las 5 con más)
# Pista SQL: FROM espigas … GROUP BY ensayo … ORDER BY n DESC … LIMIT 5
try:
    con
    espigas = pd.read_sql("SELECT * FROM espigas", con)
    mia_sql = pd.read_sql("""
        -- TU CONSULTA AQUÍ --
        SELECT ensayo, COUNT(*) AS n FROM espigas GROUP BY ensayo ORDER BY n DESC LIMIT 5
    """, con)   # ⬆ bórrala y escríbela tú; esta es para que la celda no truene sola
    gemela_pandas = espigas.groupby("ensayo").size().nlargest(5)
    print(mia_sql, "\n\n", gemela_pandas)
except NameError:
    print("⏳ Corre la celda de las gemelas 1–2 primero.")

In [ ]:
# 🧠 Gemelas 4 y 5 — TUYAS:
#   4 · espigas TOTALES por clase de estímulo (¡JOIN!):   espigas ⨝ ensayos por «ensayo»
#   5 · tasa (Hz) en P2 de cada ensayo: cuenta espigas s.t BETWEEN col26 AND col27,
#       divide entre (col27 − col26). Pista: el JOIN otra vez, y GROUP BY ensayo.
try:
    con
    q4 = pd.read_sql("""
        SELECT e.clase, COUNT(*) AS espigas
        FROM espigas s JOIN ensayos e USING (ensayo)
        GROUP BY e.clase
    """, con)   # ⬅ TU VERSIÓN AQUÍ (esta es la guía; escríbela sin mirar)
    print(q4.head(3))
    q5 = pd.read_sql("""
        SELECT e.ensayo, COUNT(s.t) / (e.col27 - e.col26) AS hz_P2
        FROM ensayos e JOIN espigas s USING (ensayo)
        WHERE s.t BETWEEN e.col26 AND e.col27
        GROUP BY e.ensayo LIMIT 5
    """, con)
    print("\n", q5)
    print("\n🤔 Para tu gemela de pandas de la 5: ¿no es esto exactamente tu en_ventana()?")
except NameError:
    print("⏳ Corre las celdas anteriores primero.")

*▶ Volvemos a las diapositivas: BD-03 — del dato crudo a la feature.*

---
# 🔵 Etapa 4 — La tabla de features: tasa por época

*📽 Vienes de BD-03.*

Las épocas del ensayo, medidas desde **t0 = fin de P1 (col25)** — la arqueología viene de
la clase de tasa de disparo:

| época | ventana | quién la define |
|---|---|---|
| `basal` | −4 → −2 s | fija |
| `P1` | col24 → col25 | cada ensayo |
| `demora1` | 0 → 2.01 s | fija |
| `P2` | col26 → col27 | cada ensayo |
| `demora2` | col27 → col28 | fija |
| `post_retro` ⚠ | col30 → +1 s | **después de la recompensa** 💸 |

Y la fábrica de features **ya la escribiste**: `CalculadorDeTasas.en_ventana()`, tu worker
de la Clase 2, ahora produciendo ciencia.

In [ ]:
# La tabla `tasas`: 140 ensayos × 6 épocas, con TU worker
from neurona import CalculadorDeTasas

try:
    espigas_por_ensayo = Cargador(CSV).tiempos_de_espiga()
    calc = CalculadorDeTasas()
    t0 = meta[:, 25]
    ev = meta[:, 20:31] - t0[:, None]          # eventos relativos a t0, por ensayo

    tasas = pd.DataFrame({"ensayo": meta[:, 0].astype(int),
                          "clase": meta[:, 1].astype(int),
                          "acierto": meta[:, 4].astype(int)})
    for i, e in enumerate(espigas_por_ensayo):
        e = np.asarray(e) - t0[i]              # espigas también relativas a t0
        r = ev[i]
        tasas.loc[i, "basal"]      = calc.en_ventana(e, -4.0, -2.0)
        tasas.loc[i, "P1"]         = calc.en_ventana(e, r[4], r[5])
        tasas.loc[i, "demora1"]    = calc.en_ventana(e, 0.0, 2.01)
        tasas.loc[i, "P2"]         = calc.en_ventana(e, r[6], r[7])
        tasas.loc[i, "demora2"]    = calc.en_ventana(e, r[7], r[8])
        tasas.loc[i, "post_retro"] = calc.en_ventana(e, r[10], r[10] + 1.0)

    tasas.to_sql("tasas", con, if_exists="replace", index=False)
    print("✔ tabla `tasas` en neurona.sqlite:", tasas.shape)
    print(tasas.head(3).round(2))
except NotImplementedError as pista:
    print("⏳ Necesita tus workers (Cargador y CalculadorDeTasas). Pista:", pista)
except NameError:
    print("⏳ Corre antes las etapas 2 y 3 (necesito `meta` y `con`).")

In [ ]:
# ¿En qué época separa la neurona? Aciertos vs errores, época por época
import matplotlib.pyplot as plt

try:
    epocas = ["basal", "P1", "demora1", "P2", "demora2", "post_retro"]
    fig, axes = plt.subplots(1, 6, figsize=(15, 3.2), sharey=True)
    for ax, ep in zip(axes, epocas):
        datos = [tasas.loc[tasas.acierto == 1, ep], tasas.loc[tasas.acierto == 0, ep]]
        ax.boxplot(datos, tick_labels=["acierto", "error"])
        ax.set_title(ep + (" ⚠" if ep == "post_retro" else ""))
    axes[0].set_ylabel("tasa (Hz)")
    plt.tight_layout(); plt.show()
    print("Mira P2… y mira la última. De la última hablamos la próxima clase 💸")
except NameError:
    print("⏳ Corre antes la celda de la tabla `tasas`.")

### 🎯 Formular la pregunta (esto queda ANOTADO para la próxima clase)

* **Unidad de análisis:** el ensayo (n = 140).
* **`y`** = acierto del mono (col4): **109 aciertos, 31 errores**.
* **`X`** = las tasas por época de tu tabla `tasas`.
* **El número a vencer: 77,9 %.** Un «modelo» que diga siempre *acierto* ya acierta eso
  sin aprender nada. La próxima clase, cualquier resultado se compara contra ESE número.

> 🧾 **Pagaré 💸 — se cobra la próxima clase:** en tu tabla quedó la columna
> `post_retro`, medida DESPUÉS de la recompensa. ¿Es legítimo usarla para predecir el
> acierto? Piénsalo hoy; la respuesta, con datos, el miércoles.

*▶ Volvemos a las diapositivas: TAL-03 (configuración y secretos) y API-01.*

---
# 🟣 Etapa 5 — Tu código le habla a GitHub

*📽 Vienes de API-01 y TAL-03.*

**5a · Las dependencias cambian con el proyecto.** En la terminal:

```
pip install requests python-dotenv
```

y añade estas dos líneas al final de `requirements.txt` (¡edítalo en VSCode!):

```
requests>=2.31
python-dotenv>=1.0
```

Commit: `git add requirements.txt` → `git commit -m "El proyecto aprende a hablar con APIs: requests y dotenv"` → `git push`.

**5b · El token a su casa.** Crea un archivo **`.env`** en la raíz del proyecto con una
sola línea (el token que creaste en casa — guía en `docs/token-github.md`):

```
GITHUB_TOKEN=github_pat_xxxxxxxxxxxx
```

Corre `git status`: **el `.env` no debe aparecer.** Esa ausencia es un secreto bien guardado.

In [ ]:
# Tu trabajo del miércoles, de regreso desde la nube ☁️
import os

try:
    import requests
    from dotenv import load_dotenv
except ImportError:
    print("⏳ Instala primero: pip install requests python-dotenv  (Etapa 5a)")
else:
    load_dotenv()
    token = os.environ.get("GITHUB_TOKEN", "")
    if not token:
        print("⏳ Falta el .env con GITHUB_TOKEN=…  (Etapa 5b — guía en docs/token-github.md)")
    else:
        USUARIO = "TU-USUARIO"       # ⬅ cámbialo por el tuyo
        headers = {"Authorization": f"Bearer {token}"}
        r = requests.get(f"https://api.github.com/repos/{USUARIO}/neurona", headers=headers)
        print("status:", r.status_code)
        if r.status_code == 200:
            repo = r.json()
            print("repo:", repo["full_name"], "· issues abiertos:", repo["open_issues_count"])
            ri = requests.get(f"https://api.github.com/repos/{USUARIO}/neurona/issues",
                              params={"state": "closed"}, headers=headers)
            print("\nTus issues CERRADOS (tu historia, contada por la API):")
            for issue in ri.json():
                print(f"  #{issue['number']} · {issue['title']}")
        elif r.status_code == 404:
            print("404: revisa USUARIO (arriba) — ¿así se llama tu cuenta?")
        elif r.status_code in (401, 403):
            print("Sin permiso o sin cuota: revisa el token del .env (¿lo copiaste completo?)")

---
## ✅ Checkpoint de la clase 🔮

Sin ejecutar ni mirar:

1. ¿Por qué la neurona necesitó DOS tablas y no una? ¿Cómo se llaman y qué une al JOIN?
2. `git status` no muestra ni `neurona.sqlite` ni `.env`. ¿Por la misma razón? ¿Cuál es la de cada uno?
3. `SELECT clase, COUNT(*) FROM ensayos GROUP BY clase` — dilo en pandas. Y al revés: `ensayos.acierto.mean()` en SQL.
4. En la tabla `tasas`, ¿qué es una fila? ¿Y una columna? ¿Por qué eso es «ingeniería de features»?
5. Sin token la API da 60 peticiones/hora **por red**; con token, 5 000 **por persona**. ¿Por qué el aula rompe lo primero?
6. El 77,9 % a vencer: ¿de dónde sale ese número exactamente?

> 💬 **Para tu reporte (Clase 3):** el diccionario dejó las columnas 5–19 como
> «desconocida». Defiende esa decisión: ¿qué riesgos evita, comparado con adivinar? ¿Y qué
> te dice sobre los datasets «bien documentados» que descargarás en tu carrera? *(2–5
> renglones honestos en `docs/reporte.md`; declara ahí tus usos de IA.)*

---
# 🏆 Proyecto integrador (continúa)

Rúbrica: **funciona 70 % · comentado 15 % · discusión 15 %**.

**Hoy sales con:** diccionario commiteado (Closes #4) · `neurona.sqlite` regenerable ·
tabla `tasas` construida · dependencias nuevas versionadas · la API respondiéndote.

**Tarea (al martes 13 — hay una semana):** las gemelas que faltaron (con su gemela de
pandas) · el diccionario terminado si quedó a medias · y tu 💬 en `docs/reporte.md`.

---
# 🔭 ¿Qué sigue?

La pregunta ya está formulada y el número a vencer, anotado: **77,9 %**. La próxima clase
entrenamos el primer modelo… y desmontamos una ilusión: verás un modelo «con 77 % de
exactitud» que en realidad **no aprendió nada**, y una «mejora» al 81 % que es **trampa
pura** (tu pagaré 💸 de la columna `post_retro`). Va a doler bonito. 🧠

---
# 🔑 Apéndice: soluciones sugeridas (gemelas 3–5)

> **Nota para el docente:** eliminar antes de distribuir si se prefiere. Cada solución es
> una forma válida entre varias.

**3 ·** SQL: `SELECT ensayo, COUNT(*) AS n FROM espigas GROUP BY ensayo ORDER BY n DESC LIMIT 5`
· pandas: `espigas.groupby("ensayo").size().nlargest(5)`

**4 ·** SQL: `SELECT e.clase, COUNT(*) AS espigas FROM espigas s JOIN ensayos e USING (ensayo) GROUP BY e.clase`
· pandas: `espigas.merge(ensayos[["ensayo","clase"]], on="ensayo").groupby("clase").size()`

**5 ·** SQL: `SELECT e.ensayo, COUNT(s.t)/(e.col27-e.col26) AS hz FROM ensayos e JOIN espigas s USING (ensayo) WHERE s.t BETWEEN e.col26 AND e.col27 GROUP BY e.ensayo`
· pandas: para cada ensayo, `calc.en_ventana(espigas_de_ese_ensayo, col26, col27)` — tu worker ES la gemela.